# 02 - Frozen features and evaluation

Takes the `proj_out` arrays from notebook 01 through BrainOmni's transformer blocks, pools them into
one feature vector per subject, and evaluates AD vs control with 5-fold cross-validation.

This mirrors what `00_extract_frozen_features` and `01_baselines` do for the sensor-level data, so
the two are directly comparable in method (though not in cohort, see the caveat below).

**Frozen only.** No fine-tuning here. Fine-tuning cost ~40 hours on the sensor data and made results
*worse*, so there is no case for spending that before knowing whether frozen features on source-space
data produce anything sensible at all.

**Interpretation caveat.** These are 161 subjects (77 AD / 84 control), not the 148 from OSF, and
they may partly overlap. So the number here is not directly comparable to the 0.831 sensor-level
result. The question this answers is narrower: *does BrainOmni produce usable representations from
parcellated source-space input with reconstructed geometry?*

In [8]:
import os, sys, json
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, torch
import torch.nn.functional as F
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
from brainomni.model import BrainOmni

OUT = '../results_parcel'
CKPT_ROOT = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection'
MODEL, SUFFIX = 'tiny', 'cortical'      # must match notebook 01

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')

proj1  = np.load(f'{OUT}/parcel_proj_out1_{MODEL}_{SUFFIX}.npy')
proj2  = np.load(f'{OUT}/parcel_proj_out2_{MODEL}_{SUFFIX}.npy')
labels = np.load(f'{OUT}/parcel_labels_{SUFFIX}.npy')
N = len(labels)
print(f'proj_out: {proj1.shape} | {N} subjects | AD {int((labels==1).sum())} / Control {int((labels==0).sum())}')
print(f'majority-class baseline: {max((labels==1).mean(), (labels==0).mean()):.3f}')

proj_out: (161, 16, 160, 256) | 161 subjects | AD 77 / Control 84
majority-class baseline: 0.522


In [9]:
cfg  = json.load(open(f'{CKPT_ROOT}/{MODEL}/model_cfg.json'))
ckpt = torch.load(f'{CKPT_ROOT}/{MODEL}/BrainOmni.pt', map_location='cpu', weights_only=True)
model = BrainOmni(**cfg); model.load_state_dict(ckpt, strict=False)
for p in model.parameters():
    p.requires_grad = False
model = model.to(device).eval()

def extract_frozen(p1, p2):
    """proj_out -> transformer blocks -> L2 normalize -> mean over time -> average the two
    segments -> flatten. Same definition used for the sensor-level frozen features."""
    feats = []
    with torch.no_grad():
        for i in range(N):
            x1 = torch.tensor(p1[i]).unsqueeze(0).to(device)
            x2 = torch.tensor(p2[i]).unsqueeze(0).to(device)
            for b in list(model.blocks)[:-1]:
                x1 = b(x1); x2 = b(x2)
            x1 = F.normalize(x1, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
            x2 = F.normalize(x2, p=2.0, dim=-1, eps=1e-6).mean(dim=2)
            feats.append(((x1 + x2) / 2).flatten().cpu().numpy())
    return np.stack(feats)

X = extract_frozen(proj1, proj2)
np.save(f'{OUT}/parcel_frozen_{MODEL}_{SUFFIX}.npy', X)
print('frozen features:', X.shape)
print('finite?', bool(np.isfinite(X).all()), '| range', float(X.min()), 'to', float(X.max()))

/opt/miniconda3/envs/brainomni/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


frozen features: (161, 4096)
finite? True | range -0.3510485291481018 to 0.23505078256130219


In [10]:
# 5-fold CV, seed 123, same protocol as the sensor-level baselines.
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(5, shuffle=True, random_state=123)
r  = cross_validate(LogisticRegression(max_iter=2000), X, labels, cv=cv,
                    scoring=['accuracy', 'roc_auc'])
acc, auc = r['test_accuracy'].mean(), r['test_roc_auc'].mean()
print(f'Frozen BrainOmni-{MODEL} on parcels ({SUFFIX}):')
print(f'  accuracy {acc:.3f}   (per fold: {np.round(r["test_accuracy"], 3)})')
print(f'  AUC      {auc:.3f}   (per fold: {np.round(r["test_roc_auc"], 3)})')

Frozen BrainOmni-tiny on parcels (cortical):
  accuracy 0.850   (per fold: [0.939 0.812 0.875 0.812 0.812])
  AUC      0.922   (per fold: [0.996 0.848 0.918 0.914 0.933])


In [11]:
# Decision gate: is this working, or is something broken upstream?
chance = max((labels == 1).mean(), (labels == 0).mean())
print(f'majority-class baseline : {chance:.3f}')
print(f'this result             : {acc:.3f}  (AUC {auc:.3f})')
print()
if acc < chance + 0.05:
    print('NEAR CHANCE. Something upstream is likely broken rather than this being a real')
    print('null result. Worth checking before interpreting:')
    print('  - sign flipping: if not applied, regions are inverted in a random subset of subjects')
    print('  - geometry: re-check the centroid plot in notebook 00')
    print('  - feature range: compare against results_withseed/frozen_base.npy')
elif acc < 0.70:
    print('WEAK. Above chance but well below the sensor-level result, consistent with')
    print('source-space input being out of distribution for a sensor-pretrained model.')
else:
    print('WORKING. BrainOmni produces usable representations from parcellated input')
    print('with reconstructed geometry. Fine-tuning would now be worth considering.')
print()
print('For reference, sensor-level frozen BrainOmni-base on the OSF cohort reached 0.831 / 0.878,')
print('but that is a different cohort (148 vs 161 subjects), so treat it as context, not a')
print('like-for-like comparison.')

majority-class baseline : 0.522
this result             : 0.850  (AUC 0.922)

WORKING. BrainOmni produces usable representations from parcellated input
with reconstructed geometry. Fine-tuning would now be worth considering.

For reference, sensor-level frozen BrainOmni-base on the OSF cohort reached 0.831 / 0.878,
but that is a different cohort (148 vs 161 subjects), so treat it as context, not a
like-for-like comparison.
